# BIST basket monthly rebalance — chart-first v13

Bu sürüm özellikle **okunabilirlik ve karşılaştırma** için hazırlandı.

İçerik:
- veri kapsamı grafikleri
- benchmark piyasa grafikleri
- korelasyon ısı haritası
- aylık yeniden kurulan portföy
- hangi varlıkların seçildiği
- ağırlık evrimi
- performans, drawdown ve aylık getiri grafikleri

Dürüst not:
- Aşağıdaki BIST listesi önceki geniş evren not defterinden alınmış başlangıç listesidir.
- Bunu canlı doğrulanmış güncel resmi BIST100 listesi gibi sunmuyorum.


In [ ]:
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

from pypfopt import expected_returns, risk_models, EfficientFrontier

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 1) Universe config

Burada BIST hisseleri, döviz ve kıymetli madenler birlikte denenir.
TEFAS şimdilik manuel / opsiyoneldir.


In [ ]:
BIST_TICKERS = ['AEFES.IS', 'AGHOL.IS', 'AKBNK.IS', 'AKFYE.IS', 'AKSA.IS', 'AKSEN.IS', 'ALARK.IS', 'ARCLK.IS', 'ASELS.IS', 'ASTOR.IS', 'BIMAS.IS', 'BOBET.IS', 'CCOLA.IS', 'CIMSA.IS', 'CWENE.IS', 'DOAS.IS', 'DOHOL.IS', 'ECILC.IS', 'EGEEN.IS', 'EKGYO.IS', 'ENERY.IS', 'ENJSA.IS', 'ENKAI.IS', 'EREGL.IS', 'EUPWR.IS', 'FROTO.IS', 'GARAN.IS', 'GESAN.IS', 'GUBRF.IS', 'HALKB.IS', 'HEKTS.IS', 'ISCTR.IS', 'ISMEN.IS', 'KAYSE.IS', 'KCAER.IS', 'KCHOL.IS', 'KLSER.IS', 'KOZAA.IS', 'KOZAL.IS', 'KRDMD.IS', 'MAVI.IS', 'MGROS.IS', 'MIATK.IS', 'ODAS.IS', 'OTKAR.IS', 'OYAKC.IS', 'PETKM.IS', 'PGSUS.IS', 'REEDR.IS', 'SASA.IS', 'SAHOL.IS', 'SDTTR.IS', 'SISE.IS', 'SKBNK.IS', 'SMRTG.IS', 'SOKM.IS', 'TAVHL.IS', 'TCELL.IS', 'THYAO.IS', 'TKFEN.IS', 'TOASO.IS', 'TSKB.IS', 'TTKOM.IS', 'TTRAK.IS', 'TUPRS.IS', 'ULKER.IS', 'VAKBN.IS', 'VESBE.IS', 'VESTL.IS', 'YKBNK.IS', 'ZOREN.IS']

FX_TICKERS = {'USDTRY': 'USDTRY=X', 'EURTRY': 'EURTRY=X'}
METAL_USD_TICKERS = {'ALTIN_USD': 'GC=F', 'GUMUS_USD': 'SI=F', 'PLATIN_USD': 'PL=F'}
TEFAS_FUND_TICKERS = []

LOOKBACK_DAYS = 252
LOOKBACK_MONTHS = 12
MAX_WEIGHT = 0.10
MIN_HISTORY_DAYS = 180
DOWNLOAD_PERIOD = '10y'

print('BIST başlangıç evreni:', len(BIST_TICKERS))


## 2) Helpers


In [ ]:
def safe_asset_name(symbol):
    return symbol.replace('.IS', '').replace('=X', '').replace('=F', '')

def tr_money(x):
    return f'₺{x:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def fetch_close_series(symbol, period=DOWNLOAD_PERIOD):
    df = yf.download(symbol, period=period, auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs('Close', axis=1, level=0) if 'Close' in level0 else df.iloc[:, :1]
    else:
        close = df[['Close']] if 'Close' in df.columns else df.iloc[:, :1]
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]
    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def normalize_prices(df):
    return df / df.iloc[0] * 100

def clean_returns(df):
    out = df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='any')
    valid = [c for c in out.columns if out[c].std() > 1e-10 and np.isfinite(out[c]).all()]
    return out[valid] if len(valid) else pd.DataFrame(index=out.index)


## 3) Data loading and coverage

Önce neyin gerçekten indirildiğini ve testte kalabildiğini görelim.


In [ ]:
rows = []
series_list = []

# BIST equities
for sym in BIST_TICKERS:
    s = fetch_close_series(sym)
    rows.append({'source': sym, 'asset': safe_asset_name(sym), 'group': 'BIST', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})
    if s is not None and len(s) >= MIN_HISTORY_DAYS:
        s = s.copy(); s.name = safe_asset_name(sym); series_list.append(s)

fx_series = {}
for label, sym in FX_TICKERS.items():
    s = fetch_close_series(sym)
    fx_series[label] = s
    rows.append({'source': sym, 'asset': label, 'group': 'FX', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})
    if s is not None and len(s) >= MIN_HISTORY_DAYS:
        s = s.copy(); s.name = label; series_list.append(s)

metal_usd = {}
for label, sym in METAL_USD_TICKERS.items():
    s = fetch_close_series(sym)
    metal_usd[label] = s
    rows.append({'source': sym, 'asset': label, 'group': 'METAL_USD', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})

usdtry = fx_series.get('USDTRY')
for try_label, usd_label in [('ALTIN_TRY', 'ALTIN_USD'), ('GUMUS_TRY', 'GUMUS_USD'), ('PLATIN_TRY', 'PLATIN_USD')]:
    base = metal_usd.get(usd_label)
    if base is not None and usdtry is not None:
        s = (base * usdtry).dropna()
        s.name = try_label
        rows.append({'source': f'{usd_label} * USDTRY', 'asset': try_label, 'group': 'METAL_TRY', 'n': len(s), 'start': str(s.index.min().date()), 'end': str(s.index.max().date()), 'status': 'ok'})
        if len(s) >= MIN_HISTORY_DAYS:
            series_list.append(s)

coverage = pd.DataFrame(rows)
display(coverage.head())

prices = pd.concat(series_list, axis=1).sort_index().ffill()
common_start = prices.apply(lambda s: s.first_valid_index()).max()
prices = prices.loc[common_start:].dropna()
returns = prices.pct_change().dropna()

eligible_assets = pd.DataFrame({'asset': prices.columns, 'group': [coverage.set_index('asset').loc[a, 'group'] if a in coverage['asset'].values else 'OTHER' for a in prices.columns]})
print('Ortak başlangıç:', common_start)
print('Kullanılan varlık sayısı:', prices.shape[1])
display(prices.tail())


In [ ]:
coverage_counts = coverage.groupby(['group', 'status']).size().reset_index(name='count')
fig = px.bar(coverage_counts, x='group', y='count', color='status', barmode='group', title='Veri kapsamı: grup bazında indirilen seriler')
fig.show()

eligible_counts = eligible_assets.groupby('group').size().reset_index(name='eligible_count')
fig = px.bar(eligible_counts, x='group', y='eligible_count', title='Teste kalan varlık sayısı')
fig.show()


## 4) Market benchmark charts

Burada bütün varlıkları değil, yorumlaması kolay bir benchmark setini izliyoruz:
- BIST basket (eşit ağırlıklı)
- USDTRY
- EURTRY
- ALTIN_TRY
- GUMUS_TRY
- PLATIN_TRY


In [ ]:
bist_cols = [c for c in prices.columns if c not in ['USDTRY', 'EURTRY', 'ALTIN_TRY', 'GUMUS_TRY', 'PLATIN_TRY']]
bist_basket = (1 + returns[bist_cols].mean(axis=1)).cumprod() if len(bist_cols) else pd.Series(dtype=float)
if len(bist_basket):
    bist_basket.name = 'BIST_BASKET'
bench = pd.concat([bist_basket, prices[['USDTRY', 'EURTRY', 'ALTIN_TRY', 'GUMUS_TRY', 'PLATIN_TRY']]], axis=1).dropna()
bench_norm = normalize_prices(bench.tail(504))
fig = px.line(bench_norm, x=bench_norm.index, y=bench_norm.columns, title='Son ~2 yıl benchmark görünümü (normalize)')
fig.show()

bench_rets = bench.pct_change().dropna()
fig = px.imshow(bench_rets.corr(), text_auto=True, aspect='auto', zmin=-1, zmax=1, color_continuous_scale='RdBu_r', title='Benchmark korelasyon ısı haritası')
fig.show()


## 5) Monthly portfolio builder

Bu notebookte amaç metodoloji son halini savunmak değil; aylık yeniden kurulan portföyün ne seçtiğini ve nasıl davrandığını anlaşılır biçimde göstermek.


In [ ]:
def build_portfolio(train_returns):
    train_returns = train_returns.dropna(axis=1, how='all').dropna(how='any')
    if train_returns.shape[1] < 2:
        raise ValueError('Need at least 2 assets in training window.')

    prices_window = (1 + train_returns).cumprod()
    mu = expected_returns.mean_historical_return(prices_window, frequency=252)
    S = risk_models.CovarianceShrinkage(prices_window).ledoit_wolf()

    ef = EfficientFrontier(mu, S, weight_bounds=(0, MAX_WEIGHT))
    ef.max_sharpe()
    weights = ef.clean_weights()
    weights = {k: float(v) for k, v in weights.items() if v is not None and float(v) > 0}
    return list(weights.keys()), weights


## 6) Walk-forward monthly rebalance

Gerçek hayata daha yakın kısım burası:
- her ay baştan portföy kurulur
- sadece o tarihe kadar olan veri kullanılır
- sonra bir ay elde tutulur


In [ ]:
rebal_dates = returns.resample('M').last().index
selected_history = []
weight_history = []
portfolio_rets = []

for i in range(LOOKBACK_MONTHS, len(rebal_dates) - 1):
    rebalance_date = rebal_dates[i]
    next_rebalance_date = rebal_dates[i + 1]

    train = returns.loc[:rebalance_date].tail(LOOKBACK_DAYS)
    valid_cols = train.columns[train.notna().sum() > max(60, LOOKBACK_DAYS // 3)]
    train = train[valid_cols].dropna(how='any')

    if train.empty or train.shape[1] < 2:
        continue

    try:
        chosen_assets, weights = build_portfolio(train)
    except Exception as e:
        print(f'Skipped {rebalance_date.date()} due to optimizer error: {e}')
        continue

    selected_history.append(pd.Series(chosen_assets, name=rebalance_date))
    weight_history.append(pd.Series(weights, name=rebalance_date))

    hold = returns.loc[(returns.index > rebalance_date) & (returns.index <= next_rebalance_date), list(weights.keys())].copy()
    if hold.empty:
        continue

    hold = hold.fillna(0.0)
    aligned_w = pd.Series(weights).reindex(hold.columns).fillna(0.0)
    realized = hold.mul(aligned_w, axis=1).sum(axis=1)
    portfolio_rets.append(realized)

portfolio_rets = pd.concat(portfolio_rets).sort_index()
selected_df = pd.DataFrame(selected_history)
weights_df = pd.DataFrame(weight_history).fillna(0.0)
equity_curve = (1 + portfolio_rets).cumprod()
drawdown = equity_curve / equity_curve.cummax() - 1
monthly_rets = portfolio_rets.resample('M').apply(lambda x: (1 + x).prod() - 1)

display(selected_df.tail())
display(weights_df.tail())


## 7) Selection and weight charts

Bu bölümde sistemin neyi seçtiğini daha görünür hale getiriyoruz.


In [ ]:
chosen_counts = weights_df.astype(bool).sum(axis=1).reset_index()
chosen_counts.columns = ['date', 'chosen_asset_count']
fig = px.bar(chosen_counts, x='date', y='chosen_asset_count', title='Her ay seçilen varlık sayısı')
fig.show()

asset_frequency = weights_df.astype(bool).sum().sort_values(ascending=False).reset_index()
asset_frequency.columns = ['asset', 'months_selected']
fig = px.bar(asset_frequency.head(20), x='asset', y='months_selected', title='En sık seçilen 20 varlık')
fig.show()

latest_weights = weights_df.iloc[-1].sort_values(ascending=False)
latest_weights = latest_weights[latest_weights > 0].reset_index()
latest_weights.columns = ['asset', 'weight']
fig = px.bar(latest_weights.head(15), x='asset', y='weight', title='Son rebalance anındaki en yüksek ağırlıklar')
fig.show()


In [ ]:
top_assets = weights_df.sum().sort_values(ascending=False).head(15).index.tolist()
heat = weights_df[top_assets].copy()
heat.index = heat.index.strftime('%Y-%m-%d')
fig = px.imshow(heat.T, aspect='auto', title='Ağırlık evrimi ısı haritası (top 15 toplam ağırlık)')
fig.update_xaxes(title='Rebalance tarihi')
fig.update_yaxes(title='Varlık')
fig.show()


## 8) Performance charts

Burada artık sonucun anlaşılır olup olmadığına bakıyoruz.


In [ ]:
ann_ret = (1 + portfolio_rets).prod() ** (252 / len(portfolio_rets)) - 1
ann_vol = portfolio_rets.std() * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol != 0 else np.nan
max_dd = drawdown.min()
summary = pd.DataFrame({'metric': ['CAGR', 'Volatility', 'Sharpe', 'Max Drawdown', 'Rebalance Count'], 'value': [ann_ret, ann_vol, sharpe, max_dd, len(weights_df)]})
display(summary)

fig = px.line(x=equity_curve.index, y=equity_curve.values, labels={'x': 'Date', 'y': 'Growth of 1'}, title='Equity curve')
fig.show()

fig = px.area(x=drawdown.index, y=drawdown.values, labels={'x': 'Date', 'y': 'Drawdown'}, title='Drawdown')
fig.show()

monthly_df = monthly_rets.reset_index()
monthly_df.columns = ['date', 'monthly_return']
fig = px.bar(monthly_df, x='date', y='monthly_return', title='Aylık portföy getirileri')
fig.show()


## 9) Raw tables

Grafiklerden sonra detay tabloyu görmek isteyenler için.


In [ ]:
print('Chosen assets each month:')
display(selected_df)
print('Weights each month:')
display(weights_df.round(4))
